### Example for running a planet search in the cloud

In [ ]:
from corazon import run_pipeline
import pandas as pd
import coiled
import ast

To download the csv files of the s3_paths and other target info, go to https://stsci.box.com/s/oxajiv21n7zqxrh1jtp4q198hpmj9em9

In [ ]:
csv_file = 'koi_or_EB_s3_paths.csv'

# All ~190k KICs observed by TESS in sectors 14-26: 'kic_s3_paths.csv' 
# The ~9k KICs that have known transits associated via KOI or Villanova EB: 'koi_or_EB_s3_paths.csv'
# The ~180k KICs that don't have a transit in KOI or Villanova EB: 'kic_no_transits_s3_paths.csv'

df = pd.read_csv(csv_file) # read in the file
df['s3_paths'] = df['s3_paths'].apply(ast.literal_eval) # make the s3_path and sector lists and not just strings with '[]' characters in them
df['sectors'] = df['sectors'].apply(ast.literal_eval)

# Make a list of lists to throw into the planet search
tics = df['TIC']
s3_paths = df['s3_paths']

targetinfo = []
for i, row in enumerate(df['s3_paths']):
    tic = df['TIC'][i]
    for s3_path in row: 
        s3p = s3_path
        info = [tic, s3p, int(s3p[33:35])]
        targetinfo.append(info)

In [ ]:
# use coiled to decorate a BLS search using corazon to run in the cloud

@coiled.function()
def wrapped_runone_s3_search(tinfo):
    try:
        run_tag =  '20250428'
        lc_author = 'TGLC'
        out_dir = 'corazon-search-2025/search-test'
        sector = tinfo[2]
        tic = tinfo[0]
        s3_path = tinfo[1]
        config = {
                "max_period_days": 12,
                "min_period_days": 0.8,
                "bls_durs_hrs": [1, 2, 4, 8, 12],
                "minSnr": [1],
                "maxTces": 6,
                "fracRemain": 0.7
            }
        result = corazon.run_pipeline.run_write_one_coiled(ticid=tic, s3_location=s3_path, run_tag=run_tag, 
                                                           lc_author=lc_author, sector=sector, config_file=config,
                                                           local=False, out_dir=out_dir, plot=False)

    except Exception as e:
        return str(e)
    
    return result

In [ ]:
# Run that search using Coiled's .map to spin up a bunch of AWS servers
search_result = list(wrapped_runone_s3_search.map(targetinfo[1000:10000]))